In [ ]:
import os, glob, torch

hits = glob.glob('/kaggle/input/**/train.csv', recursive=True)
print('train.csv found at:', hits)

BASE = os.path.dirname(hits[0])
print('BASE =', BASE)
print('train wavs:', len(os.listdir(f'{BASE}/train')), '| test wavs:', len(os.listdir(f'{BASE}/test')))
print('cuda available:', torch.cuda.is_available())

In [ ]:
# A1
!pip install -q faster-whisper

import os, re, json, time, glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import spearmanr

sns.set_style('whitegrid')
BASE = '/kaggle/input/competitions/shl-hiring-assessment-2026/Dataset_Final'
WORK = '/kaggle/working'
ASR_DIR = f'{WORK}/asr_raw'
os.makedirs(ASR_DIR, exist_ok=True)

train = pd.read_csv(f'{BASE}/train.csv')
test = pd.read_csv(f'{BASE}/test.csv')
uids = ['train/' + f for f in train.filename] + ['test/' + f for f in test.filename]
print(len(uids), 'files')

In [ ]:
from kaggle_secrets import UserSecretsClient
import os

# Retrieve the token from Kaggle Secrets
user_secrets = UserSecretsClient()
hf_token = user_secrets.get_secret("HF_TOKEN")  # Changed variable name here

# Set it as an environment variable
os.environ["HF_TOKEN"] = hf_token

print("HF_TOKEN successfully loaded!")


In [ ]:
# A2
from faster_whisper import WhisperModel

model = WhisperModel('large-v3', device='cuda', compute_type='float16')

# prompt with fillers and repetitions: heuristic to make Whisper keep disfluencies (verified in the A/B cell)
PROMPT = "Umm, let me think like, hmm... Okay, here's what I'm, like, thinking. I I was going to, uh, say that."

def asr(uid, use_prompt=True):
    segs, info = model.transcribe(
        f'{BASE}/{uid}', language='en', task='transcribe', beam_size=5,
        condition_on_previous_text=False,
        initial_prompt=PROMPT if use_prompt else None,
        word_timestamps=True, vad_filter=False)
    seg_list = []
    for s in segs:
        seg_list.append(dict(
            start=s.start, end=s.end, text=s.text,
            avg_logprob=s.avg_logprob, no_speech_prob=s.no_speech_prob,
            compression_ratio=s.compression_ratio,
            words=[dict(w=w.word, s=w.start, e=w.end, p=w.probability) for w in (s.words or [])]))
    return dict(uid=uid, duration=info.duration, segments=seg_list)

FILLER_RE = re.compile(r"\b(um+|uh+|hmm+|mm+|erm?|ah+)\b")
def tokens(text):
    return re.findall(r"[a-z']+", text.lower())
def full_text(res):
    return ' '.join(s['text'].strip() for s in res['segments'])

In [ ]:
import soundfile as sf

def load_audio(uid):
    y, sr = sf.read(f'{BASE}/{uid}', dtype='float32', always_2d=False)
    if y.ndim > 1:
        y = y.mean(axis=1)
    assert sr == 16000, f'unexpected sample rate {sr} for {uid}'
    return y

def asr(uid, use_prompt=True):
    segs, info = model.transcribe(
        load_audio(uid), language='en', task='transcribe', beam_size=5,
        condition_on_previous_text=False,
        initial_prompt=PROMPT if use_prompt else None,
        word_timestamps=True, vad_filter=False)
    seg_list = []
    for s in segs:
        seg_list.append(dict(
            start=s.start, end=s.end, text=s.text,
            avg_logprob=s.avg_logprob, no_speech_prob=s.no_speech_prob,
            compression_ratio=s.compression_ratio,
            words=[dict(w=w.word, s=w.start, e=w.end, p=w.probability) for w in (s.words or [])]))
    return dict(uid=uid, duration=info.duration, segments=seg_list)

# quick smoke test on one file
r = asr('train/audio_126.wav', True)
print('segments:', len(r['segments']), '| duration:', round(r['duration'], 1))
print('words in first segment:', len(r['segments'][0]['words']) if r['segments'] else 0)

In [ ]:
# A3
pick = []
for lab in [2.0, 3.0, 4.0, 5.0]:
    pick += list(train[train.label == lab].filename.iloc[:2])
ab_uids = ['train/' + f for f in pick]
lab_map = train.set_index('filename').label

rows = []
sample_text = None
for uid in ab_uids:
    t0 = time.time()
    r1 = asr(uid, True)
    sec = time.time() - t0
    r0 = asr(uid, False)
    x1, x0 = full_text(r1), full_text(r0)
    if sample_text is None:
        sample_text = x1[:250]
    rows.append(dict(uid=uid, label=lab_map[uid.split('/')[1]], sec_prompt=round(sec, 1),
                     words_prompt=len(tokens(x1)), words_plain=len(tokens(x0)),
                     fillers_prompt=len(FILLER_RE.findall(x1.lower())),
                     fillers_plain=len(FILLER_RE.findall(x0.lower())),
                     prompt_leak=int('let me think' in x1.lower())))
ab = pd.DataFrame(rows)
print(ab.to_string(index=False))
print()
print(ab[['words_prompt', 'words_plain', 'fillers_prompt', 'fillers_plain', 'prompt_leak']].sum())
print('estimated hours for all 985 files:', round(ab.sec_prompt.mean() * 985 / 3600, 2))
print('\nsnippet (private notebook only):', sample_text)

In [ ]:
# A4
USE_PROMPT = True   # set False if the A/B cell showed no filler gain or any prompt leak

failed = []
done = 0
t0 = time.time()
for uid in uids:
    out = f'{ASR_DIR}/{uid.replace("/", "__")}.json'
    if os.path.exists(out):
        continue
    try:
        res = asr(uid, USE_PROMPT)
        with open(out, 'w') as f:
            json.dump(res, f)
        done += 1
    except Exception as e:
        failed.append((uid, repr(e)[:150]))
    if done and done % 50 == 0:
        el = time.time() - t0
        print(f'{done} done | {el/60:.1f} min elapsed | est remaining {(len(uids)-done)*el/done/60:.0f} min')

print('finished. new files:', done, '| failed:', len(failed))
print(failed[:5])
print('json files on disk:', len(glob.glob(f'{ASR_DIR}/*.json')))

In [ ]:
import shutil
shutil.make_archive(f'{WORK}/asr_raw_backup', 'zip', WORK, 'asr_raw')
print(round(os.path.getsize(f'{WORK}/asr_raw_backup.zip') / 1e6, 1), 'MB')

In [ ]:
expected = {f'{u.replace("/", "__")}.json' for u in uids}
on_disk_all = os.listdir(ASR_DIR)
on_disk = set(on_disk_all)

print('files in folder        :', len(on_disk_all))
print('expected (unique uids) :', len(expected))
print('missing                :', len(expected - on_disk), sorted(expected - on_disk)[:5])
print('extra / unexpected     :', len(on_disk - expected), sorted(on_disk - expected)[:5])

zero_size = [f for f in on_disk_all if os.path.getsize(f'{ASR_DIR}/{f}') == 0]
print('zero-byte files        :', len(zero_size), zero_size[:5])

mismatch, unreadable = [], []
for f in sorted(expected & on_disk):
    try:
        with open(f'{ASR_DIR}/{f}') as fh:
            r = json.load(fh)
    except Exception:
        unreadable.append(f)
        continue
    if r.get('uid', '').replace('/', '__') + '.json' != f:
        mismatch.append(f)
print('unreadable json        :', len(unreadable), unreadable[:5])
print('uid inside != filename :', len(mismatch), mismatch[:5])

ok = (len(on_disk_all) == len(expected) and not (expected - on_disk) and not (on_disk - expected)
      and not zero_size and not unreadable and not mismatch)
print('\nALL CHECKS PASSED' if ok else '\nPROBLEM FOUND, send me this output')

In [ ]:
recs, bad = [], []
for uid in uids:
    path = f'{ASR_DIR}/{uid.replace("/", "__")}.json'
    try:
        with open(path) as f:
            r = json.load(f)
    except Exception:
        bad.append(uid)
        continue
    text = ' '.join(s['text'].strip() for s in r['segments'])
    nsp = [s['no_speech_prob'] for s in r['segments']]
    recs.append(dict(uid=uid, split=uid.split('/')[0], duration=r['duration'],
                     n_words=len(tokens(text)), n_segments=len(r['segments']),
                     max_no_speech=max(nsp) if nsp else np.nan,
                     mean_logprob=np.mean([s['avg_logprob'] for s in r['segments']]) if nsp else np.nan))

print('unreadable json files:', len(bad), bad[:5])
if bad:
    for uid in bad:
        os.remove(f'{ASR_DIR}/{uid.replace("/", "__")}.json')
    print('deleted them. Re-run cell A4 (it only redoes the missing files), then run this cell again.')
else:
    asr_df = pd.DataFrame(recs)
    asr_df = asr_df.merge(train.assign(uid='train/' + train.filename)[['uid', 'label']], on='uid', how='left')
    asr_df.to_csv(f'{WORK}/asr_summary.csv', index=False)

    trn = asr_df[asr_df.split == 'train']
    tst = asr_df[asr_df.split == 'test']
    nz = trn[trn.label > 0]
    print('train zero-label n_words:\n', trn[trn.label == 0].n_words.describe().round(1))
    print('\ntrain non-zero n_words:\n', nz.n_words.describe().round(1))
    print('\ntest n_words:\n', tst.n_words.describe().round(1))
    rho, p = spearmanr(nz.n_words, nz.label)
    print('\nSpearman(n_words, label) on non-zero train:', round(rho, 3), 'p =', round(p, 4))
    low = tst[tst.n_words < 25]
    print('\ntest files with fewer than 25 words:', len(low))
    print(low[['uid', 'duration', 'n_words', 'max_no_speech']].to_string(index=False))

    fig, ax = plt.subplots(1, 3, figsize=(17, 4))
    sns.histplot(data=asr_df[asr_df.label.ne(0)], x='n_words', hue='split', bins=40, element='step', ax=ax[0])
    ax[0].set_title('Words per file (zeros excluded)')
    sns.scatterplot(data=nz, x='n_words', y='label', alpha=0.4, ax=ax[1])
    ax[1].set_title('Words vs label (train, non-zero)')
    sns.boxplot(data=trn.assign(is_zero=(trn.label == 0)), x='is_zero', y='n_words', ax=ax[2])
    ax[2].set_title('Words: zero vs non-zero')
    plt.tight_layout()
    plt.show()